# Признаки, обучение и валидация модели

Признаки берутся из общего модуля `features.py`Валидация — по официальной метрике из `metric.py`: **максимальный precision среди порогов с recall >= 0.70**.

Модель — CatBoost

## 1. Загрузка и признаки

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.inspection import permutation_importance

from features import build_features, events_in_window, feature_columns, load_raw

from metric import TARGET_RECALL, pr_curve, precision_at_recall, recall_at_fpr

RANDOM_STATE = 42
ROOT = Path.cwd().parent
DATA_PATH = ROOT / "data"

In [ ]:
train, test, events = load_raw(DATA_PATH)
meta, ev = events_in_window(train, test, events)
feats = build_features(meta, ev)

NUM, CAT = feature_columns(feats)
FEATURE_COLS = NUM + CAT

X_all = feats[FEATURE_COLS].copy()
X_all[CAT] = X_all[CAT].fillna("missing").astype(str)

is_train = feats["split"] == "train"
X_tr, y_tr = X_all[is_train.values], feats.loc[is_train, "target"].astype(int)
X_te = X_all[~is_train.values]
dates_tr = feats.loc[is_train, "window_start_ts"]

print(f"Признаков: {len(FEATURE_COLS)} ({len(NUM)} числовых + {len(CAT)} категориальных)")
print(f"train: {X_tr.shape}, test: {X_te.shape}, доля ботов: {y_tr.mean():.4f}")

## 2. Валидация по времени

В EDA было видно, что test идёт строго после train по датам. Случайный K-fold в этой ситуации завышает качество. Вместо этого делим train по времени: несколько блоков дней, каждый следующий блок валидируется на модели, обученной только на более ранних днях (expanding window).

In [ ]:
def time_blocks(dates: pd.Series, n_blocks: int) -> list[pd.Series]:
    """Бьёт уникальные дни на n_blocks последовательных по времени групп."""
    days = np.sort(dates.dt.date.unique())
    edges = np.array_split(days, n_blocks)
    return [dates.dt.date.isin(block) for block in edges]


def time_folds(dates: pd.Series, n_blocks: int = 5, min_train_blocks: int = 2):
    """Expanding-window фолды: обучение на всех блоках до i, валидация на блоке i."""
    blocks = time_blocks(dates, n_blocks)
    for i in range(min_train_blocks, n_blocks):
        train_mask = np.logical_or.reduce(blocks[:i])
        valid_mask = blocks[i].values
        yield train_mask, valid_mask

## 3. Модель

`CatBoostClassifier` с `cat_features=CAT` — категориальные колонки не нужно кодировать вручную, библиотека сама строит для них статистики. Классы разбалансированы (~8% ботов), поэтому `auto_class_weights="Balanced"`.

In [ ]:
def make_model():
    return CatBoostClassifier(
        iterations=600, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
        cat_features=CAT, auto_class_weights="Balanced",
        loss_function="Logloss", eval_metric="Logloss",
        random_seed=RANDOM_STATE, verbose=False, allow_writing_files=False,
    )


def fit_score(X_fit, y_fit, X_eval, y_eval=None):
    model = make_model()
    if y_eval is not None:
        model.fit(X_fit, y_fit, eval_set=(X_eval, y_eval), early_stopping_rounds=50)
    else:
        model.fit(X_fit, y_fit)
    return model.predict_proba(X_eval)[:, 1], model

## 4. Кросс-валидация по официальной метрике

In [ ]:
fold_rows = []
for i, (fit_mask, val_mask) in enumerate(time_folds(dates_tr, n_blocks=5, min_train_blocks=2), start=1):
    y_val = y_tr[val_mask]
    score_val, model = fit_score(X_tr[fit_mask], y_tr[fit_mask], X_tr[val_mask], y_val)
    fold_rows.append({
        "fold": i,
        "n_fit": int(fit_mask.sum()), "n_val": int(val_mask.sum()), "bots_val": int(y_val.sum()),
        "best_iteration": model.get_best_iteration(),
        "precision_at_recall_0.70": precision_at_recall(y_val, score_val, TARGET_RECALL),
        "recall_at_fpr_0.01": recall_at_fpr(y_val, score_val, 0.01),
    })

cv_res = pd.DataFrame(fold_rows)
display(cv_res.round(3))
print("Среднее по фолдам, precision@recall0.70:",
      round(cv_res["precision_at_recall_0.70"].mean(), 3),
      "± ", round(cv_res["precision_at_recall_0.70"].std(), 3))

## 5. Финальный holdout: кривая precision/recall

In [ ]:
*_, (fit_mask, val_mask) = time_folds(dates_tr, n_blocks=5, min_train_blocks=2)
y_val = y_tr[val_mask]
score_val, holdout_model = fit_score(X_tr[fit_mask], y_tr[fit_mask], X_tr[val_mask], y_val)

prec, rec = pr_curve(y_val, score_val)
best_p = precision_at_recall(y_val, score_val, TARGET_RECALL)
print(f"precision @ recall>=0.70 на последнем holdout-блоке: {best_p:.3f}")
print("best_iteration:", holdout_model.get_best_iteration())

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 4))
plt.plot(rec, prec, marker=".", lw=1)
plt.axvline(TARGET_RECALL, color="gray", ls="--", lw=1)
plt.xlabel("recall"); plt.ylabel("precision")
plt.title("PR-кривая, последний time-блок")
plt.tight_layout()
plt.show()

## 6. Важность признаков

Permutation importance на holdout-блоке, метрика — та же официальная `precision_at_recall`

In [ ]:
def metric_scorer(estimator, X, y):
    score = estimator.predict_proba(X)[:, 1]
    return precision_at_recall(y, score, TARGET_RECALL)

imp = permutation_importance(
    holdout_model, X_tr[val_mask], y_val, scoring=metric_scorer,
    n_repeats=5, random_state=RANDOM_STATE,
)
imp_tab = pd.DataFrame({"feature": FEATURE_COLS, "importance": imp.importances_mean,
                         "std": imp.importances_std}).sort_values("importance", ascending=False)
display(imp_tab.head(20).round(4))

cb_imp = pd.Series(holdout_model.get_feature_importance(), index=FEATURE_COLS, name="catboost_importance")
display(cb_imp.sort_values(ascending=False).head(20).round(2))

## 7. Финальное обучение и сабмит

Число итераций фиксируем по медианному `best_iteration` из CV (без eval_set на полном train — валидировать там уже не на чем). Скорим test, сохраняем `score` по `cookie_id`.

In [ ]:
final_iterations = int(cv_res["best_iteration"].median())
final_model = CatBoostClassifier(
    iterations=max(final_iterations, 50), learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
    cat_features=CAT, auto_class_weights="Balanced",
    loss_function="Logloss", random_seed=RANDOM_STATE, verbose=False, allow_writing_files=False,
)
final_model.fit(X_tr, y_tr)

test_score = final_model.predict_proba(X_te)[:, 1]
submission = pd.DataFrame({"cookie_id": feats.loc[~is_train].index, "score": test_score})

assert submission["score"].between(0, 1).all()
assert submission["cookie_id"].is_unique
assert len(submission) == len(test)

display(submission.describe())
submission.to_csv(ROOT / "submission.csv", index=False)